In [4]:
#imports
import time
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt

# Use the GPU if one is available (in Colab: Runtime > Change runtime type > T4 GPU).
# Every model and every batch of data must be moved to this device with .to(device).
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Fix the random seed so runs are repeatable (makes experiment comparisons fairer).
torch.manual_seed(42)

Using cpu device


In [5]:
# Where to store the dataset.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/datasets"
except Exception as e:
    print(f"Google Drive not available ({type(e).__name__}: {e}); using local storage.")
    DATA_ROOT = "data"

os.makedirs(DATA_ROOT, exist_ok=True)
print(f"Dataset folder: {DATA_ROOT}")

Mounted at /content/drive
Dataset folder: /content/drive/MyDrive/datasets


In [6]:
# Per-channel (R, G, B) mean and std of the CIFAR-10 training set.
# Normalize computes (pixel - mean) / std, so each channel is centered around 0 with
# roughly unit spread. Inputs on that scale make gradient descent train faster and more stably.
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# ToImage -> tensor of shape [C, H, W]; ToDtype(scale=True) -> floats in [0, 1]; then Normalize.
# Kept as separate train/test transforms so data augmentation (Model 3) can later be
# added to train_transform ONLY. The test set should never be augmented.
train_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Download training data (skipped automatically if it's already in DATA_ROOT).
training_data = datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=train_transform,
)

# Download test data. Same files as above, so this one won't download again.
test_data = datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=test_transform,
)

classes = training_data.classes  # ['airplane', 'automobile', 'bird', ...]
print(f"Train: {len(training_data)} images, Test: {len(test_data)} images, {len(classes)} classes")

  1%|          | 1.08M/170M [00:16<43:07, 65.5kB/s]


KeyboardInterrupt: 

In [ ]:
batch_size = 64

# Create data loaders.
# shuffle=True for training: the model sees batches in a new random order each epoch,
# so it can't learn anything from the order of the data. The test set doesn't need shuffling.
# num_workers loads batches in background processes so the GPU isn't left waiting.
train_dataloader = DataLoader(training_data, batch_size=batch_size, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

In [ ]:
# Sanity check: look at a few training images with their labels.
# The images are normalized, so undo it (x * std + mean) to get displayable [0, 1] values.
images, labels = next(iter(train_dataloader))
mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
std = torch.tensor(CIFAR10_STD).view(3, 1, 1)

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, img, label in zip(axes, images, labels):
    img = (img * std + mean).clamp(0, 1)
    ax.imshow(img.permute(1, 2, 0))  # matplotlib expects [H, W, C], PyTorch uses [C, H, W]
    ax.set_title(classes[label])
    ax.axis("off")
plt.show()

In [ ]:
# CrossEntropyLoss = softmax + negative log-likelihood, combined into one numerically stable step.
# It takes the raw model outputs ("logits"), so models should NOT end with a softmax layer.
loss_fn = nn.CrossEntropyLoss()


def train(dataloader, model, loss_fn, optimizer):
    """Run ONE epoch (one full pass over the training data). Returns (avg_loss, accuracy)."""
    model.train()  # turns ON training behavior for Dropout/BatchNorm (matters for Model 3)
    total_loss, correct, total = 0.0, 0, 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        pred = model(X)          # 1. forward pass: logits of shape [batch, 10]
        loss = loss_fn(pred, y)  # 2. how wrong were the predictions?

        optimizer.zero_grad()    # 3. clear gradients left over from the previous batch
        loss.backward()          # 4. backprop: compute d(loss)/d(weight) for every weight
        optimizer.step()         # 5. nudge each weight against its gradient

        # Bookkeeping. loss.item() is the batch's mean loss, so weight it by batch size.
        total_loss += loss.item() * X.size(0)
        correct += (pred.argmax(dim=1) == y).sum().item()
        total += X.size(0)

    return total_loss / total, correct / total


def evaluate(dataloader, model, loss_fn):
    """Measure loss and accuracy WITHOUT updating weights. Returns (avg_loss, accuracy)."""
    model.eval()  # turns OFF Dropout and makes BatchNorm use its learned statistics
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():  # no gradients needed here, which saves memory and time
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            total_loss += loss_fn(pred, y).item() * X.size(0)
            correct += (pred.argmax(dim=1) == y).sum().item()
            total += X.size(0)

    return total_loss / total, correct / total


def fit(model, optimizer, epochs, train_dl=train_dataloader, test_dl=test_dataloader):
    """Train for several epochs, evaluating after each one.
    Returns a history dict you can plot later and use to compare experiments."""
    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    start = time.time()

    for epoch in range(epochs):
        train_loss, train_acc = train(train_dl, model, loss_fn, optimizer)
        test_loss, test_acc = evaluate(test_dl, model, loss_fn)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        print(f"Epoch {epoch + 1:>2}/{epochs} | "
              f"train loss {train_loss:.4f}, acc {train_acc:.2%} | "
              f"test loss {test_loss:.4f}, acc {test_acc:.2%}")

    history["train_time_s"] = time.time() - start
    history["num_params"] = sum(p.numel() for p in model.parameters())
    print(f"Done in {history['train_time_s']:.1f}s, {history['num_params']:,} parameters")
    return history


def plot_history(histories):
    """Plot loss and accuracy curves. `histories` is a dict like {"MLP": hist1, "CNN": hist2}."""
    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in histories.items():
        ax_loss.plot(h["train_loss"], label=f"{name} train")
        ax_loss.plot(h["test_loss"], "--", label=f"{name} test")
        ax_acc.plot(h["train_acc"], label=f"{name} train")
        ax_acc.plot(h["test_acc"], "--", label=f"{name} test")
    ax_loss.set(title="Loss", xlabel="Epoch")
    ax_acc.set(title="Accuracy", xlabel="Epoch")
    ax_loss.legend()
    ax_acc.legend()
    plt.show()